In [1]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=12"

import sys
sys.path.append('../../src/')

import numpy as onp
import jax
import jax.numpy as jnp

from myutils import Ntime, ACFs, set_detectors
from myutils import ln_likelihood_full_jit

from scipy.linalg import toeplitz
import scipy.signal as sig

import lal
from gwpy.timeseries import TimeSeries
from other_utils import bandpass_ds, analysis_data, interp1d_jax, load_tables

jax.config.update("jax_enable_x64", True) 

/Users/kallol/Work/Misc/ringdown__2/examples/fixed-sky/../../src/myutils.py:3: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from chainconsumer import Chain, ChainConsumer, Truth, ChainConfig, PlotConfig

import numpyro
from numpyro.contrib.nested_sampling import NestedSampler
import numpyro.distributions as dist
numpyro.enable_x64()

/Users/kallol/miniconda3/envs/skyloc_ringdown/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
plt.rcParams["axes.grid"] = False

In [4]:
tgps = 1420878141.235932
tM = 0.337e-3 
tgps = tgps + 6*tM

seglen = 8
fs = 16384
fmin = 30
fmax       = 1500
event_id = "GW250114"
plot_checks = 0
T = 0.2
srate = 4096

ra = 2.33
dec = 0.190

factor = 10
seed       = 31567
t0         = 0.0

qnm1_path  = "../../data/l2/n1l2m2.dat"
qnm2_path  = "../../data/l2/n2l2m2.dat"

tM_shifted_samples_save_dir = "./GW250114-t-shift-posteriors/"

In [5]:
dH1 = TimeSeries.fetch_open_data('H1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)
dL1 = TimeSeries.fetch_open_data('L1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)


n_analyze = Ntime(srate, T)

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('H1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].location, ra, dec, tgps))
tH1 = tgps + dt_ifo

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('L1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].location, ra, dec, tgps))
tL1 = tgps + dt_ifo

In [6]:
dH1_cond = bandpass_ds(dH1, t0=tH1, ds=int(fs/srate), trim=0.25, f_min=fmin)
dL1_cond = bandpass_ds(dL1, t0=tL1, ds=int(fs/srate), trim=0.25, f_min=fmin)

In [7]:
psdH = sig.welch(
    dH1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",   
)

psdL = sig.welch(
    dL1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",  
)

In [8]:
dt_np = 1.0 / srate
Nt     = Ntime(srate=srate, T=T)
freqs_np = onp.fft.rfftfreq(Nt, d=dt_np)

fmin_eff = freqs_np[0]  if (fmin is None) else float(fmin)
fmax_eff = freqs_np[-1] if (fmax is None) else float(fmax)

i0 = int(onp.searchsorted(freqs_np, fmin_eff, side="left"))
i1 = int(onp.searchsorted(freqs_np, fmax_eff, side="right"))

freqs = jnp.asarray(freqs_np, dtype=jnp.float64)

# Prior limits
limits = [
    [30.0, 95.0],
    [0., 0.99],
    [0., 4.0],
    [0., 6.283185307179586],
    [0., 4.0],
    [0., 6.283185307179586],
    [-1., 1.],
    [0., 3.141592653589793]]

low  = onp.asarray([pair[0] for pair in limits], dtype=onp.float64)
high = onp.asarray([pair[1] for pair in limits], dtype=onp.float64)

In [9]:
psdH = interp1d_jax(jnp.asarray(psdH[0],dtype=jnp.float64), jnp.asarray(psdH[1],dtype=jnp.float64))
psdL = interp1d_jax(jnp.asarray(psdL[0],dtype=jnp.float64), jnp.asarray(psdL[1],dtype=jnp.float64))

omega_r, omega_i, omega_OT_r, omega_OT_i = load_tables(qnm1_path, qnm2_path)
tgps = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps)

rhoH, rhoL = ACFs(srate=srate, T=T, psdH=psdH, psdL=psdL, factor=factor)
covH=toeplitz(rhoH)
covL=toeplitz(rhoL)
L_H=jnp.linalg.cholesky(covH)
L_L=jnp.linalg.cholesky(covL)

resp_H_py = lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].response
resp_L_py = lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].response
resp_H = jnp.asarray(resp_H_py, dtype=jnp.float64)
resp_L = jnp.asarray(resp_L_py, dtype=jnp.float64)

set_detectors(resp_H, resp_L)

In [10]:
num_shifts = 15

t_shifts = []
H1_data_t_shifted = []
L1_data_t_shifted = []

for jj in range(num_shifts):
    dH1_analysis_data = analysis_data(dH1_cond[1], dH1_cond[0], tH1 + jj*1*tM, n_analyze)
    dL1_analysis_data = analysis_data(dL1_cond[1], dL1_cond[0], tL1 + jj*1*tM, n_analyze)

    t_shifts.append(jj*1*tM)
    H1_data_t_shifted.append(dH1_analysis_data[1])
    L1_data_t_shifted.append(dL1_analysis_data[1])

In [11]:
onp.savetxt(tM_shifted_samples_save_dir + 't_shifts.txt', t_shifts)

In [12]:
low  = jnp.asarray(low,  dtype=jnp.float64)  
high = jnp.asarray(high, dtype=jnp.float64)  

theta_fixed_sky = jnp.array([ra, onp.sin(dec)]) 

In [13]:
logZs = []
for i in range(num_shifts):
    h_H_t = H1_data_t_shifted[i]
    h_L_t = L1_data_t_shifted[i]

    def make_loglik_fn(dataH, dataL, gmst, L_H, L_L,
                    omega_r, omega_i, omega_OT_r, omega_OT_i,
                    T, srate, t0):
        def _loglik(theta):
            return ln_likelihood_full_jit(
                dataH=dataH, dataL=dataL, params=theta,
                gmst=gmst, L_H=L_H, L_L=L_L,
                omega_r=omega_r, omega_i=omega_i,
                omega_OT_r=omega_OT_r, omega_OT_i=omega_OT_i,
                T=T, srate=srate, t0=t0
            )
        return _loglik

    loglik_fn = make_loglik_fn(
        dataH=h_H_t, dataL=h_L_t,
        gmst=gmst, L_H=L_H, L_L=L_L,
        omega_r=omega_r, omega_i=omega_i,
        omega_OT_r=omega_OT_r, omega_OT_i=omega_OT_i,
        T=T, srate=srate, t0=t0
    )

    def model_fixed_sky():
        theta_free = numpyro.sample("theta", dist.Uniform(low, high))
        theta = jnp.concatenate([theta_free[:-1], theta_fixed_sky, jnp.array([theta_free[-1]])], axis=0)
        numpyro.factor("loglike", loglik_fn(theta))

    rng_key = jax.random.PRNGKey(int(seed) ^ 0xABCDEF)
    rng_run, rng_post = jax.random.split(rng_key)
    ns_fixed_sky = NestedSampler(
        model_fixed_sky,
        constructor_kwargs=dict(
            num_live_points=10000,   
            max_samples=500_000,  
            verbose=True,
        ),
        termination_kwargs=dict(
            dlogZ=0.01,         
        ),
    )
    ns_fixed_sky.run(rng_run)
    ns_fixed_sky.print_summary()
    posterior_fixed_sky = ns_fixed_sky.get_samples(rng_post, num_samples=100_000)

    onp.save(tM_shifted_samples_save_dir + 'posterior.' + event_id + '.' + str(i) + '.npy', onp.asarray(posterior_fixed_sky['theta']))
    logZs.append(ns_fixed_sky._results.log_Z_mean)


INFO:jaxns:Number of Markov-chains set to: 10000


Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None, evidence_uncert=None, live_evidence_frac=None, dlogZ=Array(0.01, dtype=float64, weak_type=True), max_samples=Array(500400, dtype=int64), max_num_likelihood_evaluations=None, log_L_contour=None, efficiency_threshold=None, rtol=None, atol=None, peak_XL_frac=None)
-------
Num samples: 5004
Num likelihood evals: 241608
Efficiency: 0.04058196681426694
log(L) contour: -4150.149956249995
log(Z) est.: -773.8082702910127 +- 0.6720531163625577
log(Z | remaining) est.: 3386.5331429649864 +- 0.9660265692687388
ESS: 0.875717384040438

-------
Num samples: 10008
Num likelihood evals: 519311
Efficiency: 0.02492348380894984
log(L) contour: -1931.7554636102384
log(Z) est.: -756.3107033976419 +- 0.8325532061114262
log(Z | remaining) est.: 1184.498731021906 +- 0.9092402719627808
ESS: 0.5000023370661535

--

INFO:jaxns:Number of Markov-chains set to: 10000


Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None, evidence_uncert=None, live_evidence_frac=None, dlogZ=Array(0.01, dtype=float64, weak_type=True), max_samples=Array(500400, dtype=int64), max_num_likelihood_evaluations=None, log_L_contour=None, efficiency_threshold=None, rtol=None, atol=None, peak_XL_frac=None)
-------
Num samples: 5004
Num likelihood evals: 241394
Efficiency: 0.040617212802051966
log(L) contour: -4127.779555780562
log(Z) est.: -745.7080503353018 +- 0.8325245360853173
log(Z | remaining) est.: 3391.7812870637163 +- 1.0537761960929013
ESS: 0.500025100485561

-------
Num samples: 10008
Num likelihood evals: 517678
Efficiency: 0.025093586943712436
log(L) contour: -1914.4944993651407
log(Z) est.: -746.2080455750506 +- 0.8325531310214567
log(Z | remaining) est.: 1176.3814938731418 +- 0.8712780983134356
ESS: 0.5000024620998786


INFO:jaxns:Number of Markov-chains set to: 10000


...

--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 23461687
samples: 170136
phantom samples: 0
likelihood evals / sample: 137.9
phantom fraction (%): 0.0%
--------
logZ=-731.393 +- 0.041
max(logL)=-716.372
H=-11.56
ESS=21887
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 61.8 +- 9.4 | 50.9 / 60.3 / 74.5 | 58.8 | 58.8
theta[1]: 0.5 +- 0.23 | 0.16 / 0.52 / 0.8 | 0.54 | 0.54
theta[2]: 0.4 +- 0.19 | 0.2 / 0.37 / 0.63 | 0.52 | 0.52
theta[3]: 3.3 +- 1.9 | 0.6 / 3.1 / 6.0 | 5.8 | 5.8
theta[4]: 0.26 +- 0.25 | 0.03 / 0.2 / 0.57 | 0.45 | 0.45
theta[5]: 3.2 +- 1.8 | 0.7 / 3.2 / 5.7 | 1.8 | 1.8
theta[6]: 0.12 +- 0.42 | -0.47 / 0.15 / 0.66 | 0.03 | 0.03
theta[7]: 1.58 +- 0.87 | 0.49 / 1.51 / 2.77 | 2.38 | 2.38
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=Non

INFO:jaxns:Number of Markov-chains set to: 10000


--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 21139043
samples: 165132
phantom samples: 0
likelihood evals / sample: 128.0
phantom fraction (%): 0.0%
--------
logZ=-728.415 +- 0.041
max(logL)=-713.91
H=-11.46
ESS=21466
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 62.1 +- 9.7 | 51.0 / 60.3 / 75.4 | 62.1 | 62.1
theta[1]: 0.5 +- 0.25 | 0.14 / 0.52 / 0.81 | 0.59 | 0.59
theta[2]: 0.37 +- 0.17 | 0.18 / 0.34 / 0.58 | 0.34 | 0.34
theta[3]: 3.0 +- 1.9 | 0.3 / 3.1 / 5.9 | 0.1 | 0.1
theta[4]: 0.23 +- 0.2 | 0.03 / 0.18 / 0.48 | 0.14 | 0.14
theta[5]: 3.2 +- 1.8 | 0.7 / 3.2 / 5.7 | 2.7 | 2.7
theta[6]: -0.15 +- 0.43 | -0.69 / -0.19 / 0.45 | -0.22 | -0.22
theta[7]: 1.65 +- 0.92 | 0.4 / 1.6 / 2.89 | 2.67 | 2.67
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess

INFO:jaxns:Number of Markov-chains set to: 10000


--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 23431976
samples: 170136
phantom samples: 0
likelihood evals / sample: 137.7
phantom fraction (%): 0.0%
--------
logZ=-726.595 +- 0.041
max(logL)=-711.437
H=-11.51
ESS=22468
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 60.3 +- 9.5 | 50.2 / 58.6 / 73.0 | 63.0 | 63.0
theta[1]: 0.47 +- 0.24 | 0.13 / 0.48 / 0.78 | 0.67 | 0.67
theta[2]: 0.38 +- 0.21 | 0.17 / 0.33 / 0.65 | 0.5 | 0.5
theta[3]: 3.1 +- 1.8 | 0.6 / 3.2 / 5.6 | 4.4 | 4.4
theta[4]: 0.31 +- 0.3 | 0.04 / 0.22 / 0.69 | 0.65 | 0.65
theta[5]: 3.1 +- 1.8 | 0.6 / 3.1 / 5.6 | 6.2 | 6.2
theta[6]: 0.38 +- 0.28 | 0.1 / 0.35 / 0.76 | 0.27 | 0.27
theta[7]: 1.46 +- 0.9 | 0.31 / 1.56 / 2.71 | 0.44 | 0.44
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None

In [14]:
logZs

[Array(-739.49449767, dtype=float64),
 Array(-736.09410701, dtype=float64),
 Array(-734.66463684, dtype=float64),
 Array(-733.29822642, dtype=float64),
 Array(-731.24992626, dtype=float64),
 Array(-730.32970136, dtype=float64),
 Array(-732.15647607, dtype=float64),
 Array(-732.80179184, dtype=float64),
 Array(-734.12408665, dtype=float64),
 Array(-732.38534932, dtype=float64),
 Array(-731.83842526, dtype=float64),
 Array(-731.2269212, dtype=float64),
 Array(-728.16949788, dtype=float64),
 Array(-726.48062022, dtype=float64),
 Array(-727.06825959, dtype=float64)]